# 04 — Planner, Router, and Specialist Orchestration

## Objective

Demonstrate and test the **planning and routing** layer of the investment research system:

**plan → route → run specialist agents → synthesize**

This notebook covers the following project requirements:

| Requirement | Where it is shown |
|---|---|
| Agent plans its research steps for a stock symbol | Section 3 — Planner Agent |
| Routing workflow: direct content to the right specialist | Section 4 — Router Agent |
| Uses tools / datasets dynamically | Sections 3 and 7 — the planner skips sources with no data, and only routed agents are called |
| Agent coordination | Sections 6–7 — Synthesis and Full Orchestration |

Notebooks `00`–`03` produce the processed datasets this notebook reads. It does **not** repeat ingestion, preprocessing, or the news chain.

The output of `orchestrate_research()` is the handoff to the Evaluator–Optimizer and memory workflow (Section 9).

## 1. Setup

Code for the agents lives in `src/`:

| Module | Role |
|---|---|
| `src/planner.py` | Planner Agent — checks the available evidence and decides the research steps |
| `src/router.py` | Router Agent — maps each plan step to a specialist |
| `src/news_agent.py` | News specialist — reads `news_research_results.csv` |
| `src/market_agent.py` | Market specialist — reads `market_data_clean.csv` |
| `src/insider_agent.py` | Insider specialist — reads `sec_form4_clean.csv` |
| `src/synthesis_agent.py` | Synthesis specialist — combines evidence into risks and catalysts |
| `src/orchestrator.py` | Coordinates the full planner → router → specialist workflow |

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.insider_agent import analyze_insider_activity
from src.market_agent import analyze_market
from src.news_agent import analyze_news
from src.orchestrator import display_orchestration, orchestrate_research
from src.planner import generate_research_plan
from src.router import route_step
from src.synthesis_agent import synthesize_research

pd.set_option("display.max_colwidth", 120)

print(f"Project root: {PROJECT_ROOT}")

## 2. Check Required Inputs

The specialist agents read processed files written by notebooks `01` and `03`. If any file is missing, run those notebooks first.

In [ ]:
PROCESSED = PROJECT_ROOT / "data" / "processed"

REQUIRED_FILES = {
    "news_agent": PROCESSED / "news_research_results.csv",
    "market_agent": PROCESSED / "market_data_clean.csv",
    "insider_agent": PROCESSED / "sec_form4_clean.csv",
}

missing = [
    f"{agent}: {path.relative_to(PROJECT_ROOT)}"
    for agent, path in REQUIRED_FILES.items()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing processed inputs. Run notebooks 00-03 first:\n"
        + "\n".join(missing)
    )

for agent, path in REQUIRED_FILES.items():
    rows = len(pd.read_csv(path))
    print(f"{agent:<14} {path.name:<28} {rows:>6} rows")

### Choose the ticker

Uses the first ticker found in the market data. Change `TICKER` to research a different symbol.

In [ ]:
available_tickers = sorted(
    pd.read_csv(REQUIRED_FILES["market_agent"])["ticker"]
    .astype(str)
    .str.upper()
    .unique()
)

print(f"Tickers with market data: {available_tickers}")

TICKER = available_tickers[0]
print(f"Researching: {TICKER}")

## 3. Planner Agent

The Planner Agent decides the research steps for a ticker. Before planning, it checks how many records each processed dataset has for that ticker:

- A data source with records gets an analysis step.
- A data source with no records is **skipped**, and the reason is recorded.
- Risk, catalyst, and final-synthesis steps are added only if at least one source has evidence.

Each decision is saved in `planning_decisions`, so the plan can be explained. Called without `project_root`, the planner returns the full six-step plan without checking data.

In [ ]:
plan = generate_research_plan(TICKER, PROJECT_ROOT)

print(plan["objective"])
print("Evidence found:", plan["evidence_counts"])
print()
print("Planner decisions:")
display(pd.DataFrame(plan["planning_decisions"]))
pd.DataFrame(plan["research_steps"])

In [ ]:
# Tests: the plan is well-formed and follows the available evidence.
steps = plan["research_steps"]

assert plan["ticker"] == TICKER
assert plan["evidence_checked"] is True
assert [s["step_id"] for s in steps] == list(range(1, len(steps) + 1))
assert all({"step_id", "task", "description"} <= s.keys() for s in steps)

# A source is planned only if it has records.
planned_tasks = {s["task"] for s in steps}
for source, task in [("news", "news_analysis"), ("market", "market_analysis"),
                     ("insider", "insider_analysis")]:
    assert (task in planned_tasks) == (plan["evidence_counts"][source] > 0), task

# Input is normalised: lower-case and padded tickers give the same plan.
assert generate_research_plan(f"  {TICKER.lower()} ", PROJECT_ROOT) == plan

# Without checking data, the planner returns the full six-step plan.
assert len(generate_research_plan(TICKER)["research_steps"]) == 6

# A ticker with no data gets no steps, and every skip has a reason.
empty_plan = generate_research_plan("NO_SUCH_TICKER", PROJECT_ROOT)
assert empty_plan["research_steps"] == []
assert all(not d["included"] and d["reason"] for d in empty_plan["planning_decisions"])

print(f"Planner tests passed ({len(steps)} steps planned for {TICKER}).")

## 4. Router Agent (Routing Workflow)

The Router reads each plan step and directs it to the specialist that owns that kind of evidence, with a reason for the choice:

| Task | Specialist |
|---|---|
| `news_analysis` | `news_agent` |
| `market_analysis` | `market_agent` |
| `insider_analysis` | `insider_agent` |
| `risk_analysis`, `catalyst_analysis`, `final_synthesis` | `synthesis_agent` |

Any task it does not recognise falls back to `synthesis_agent`.

In [ ]:
routing_decisions = [route_step(step) for step in plan["research_steps"]]

pd.DataFrame(routing_decisions)[["step_id", "task", "agent", "reason"]]

In [ ]:
# Tests: every step is routed to the expected specialist.
EXPECTED_ROUTES = {
    "news_analysis": "news_agent",
    "market_analysis": "market_agent",
    "insider_analysis": "insider_agent",
    "risk_analysis": "synthesis_agent",
    "catalyst_analysis": "synthesis_agent",
    "final_synthesis": "synthesis_agent",
}

assert len(routing_decisions) == len(plan["research_steps"])

for decision in routing_decisions:
    assert decision["agent"] == EXPECTED_ROUTES[decision["task"]], decision
    assert decision["reason"]

# Edge cases: task names are normalised, unknown tasks fall back.
assert route_step({"step_id": 1, "task": " NEWS_ANALYSIS "})["agent"] == "news_agent"
assert route_step({"step_id": 99, "task": "unknown_task"})["agent"] == "synthesis_agent"
assert route_step({})["agent"] == "synthesis_agent"

print("Router tests passed.")

## 5. Specialist Agents (Tool / Dataset Use)

Each specialist loads only the dataset it needs and returns a structured result for the ticker. If a ticker has no rows in a dataset, the agent returns a count of `0` and a message instead of failing.

In [ ]:
news_result = analyze_news(TICKER, PROJECT_ROOT)

print(f"Articles: {news_result['article_count']}")
print(f"Categories: {news_result['categories']}")

pd.DataFrame(news_result["research_notes"]).head()

In [ ]:
market_result = analyze_market(TICKER, PROJECT_ROOT)
market_result

In [ ]:
insider_result = analyze_insider_activity(TICKER, PROJECT_ROOT)
insider_result

In [ ]:
# Tests: each specialist reports on the right ticker with the expected fields.
for result, name, count_key in [
    (news_result, "news_agent", "article_count"),
    (market_result, "market_agent", "row_count"),
    (insider_result, "insider_agent", "transaction_count"),
]:
    assert result["agent"] == name
    assert result["ticker"] == TICKER
    assert result[count_key] >= 0

# Specialists handle a ticker with no data without raising.
empty_news = analyze_news("NO_SUCH_TICKER", PROJECT_ROOT)
empty_market = analyze_market("NO_SUCH_TICKER", PROJECT_ROOT)
empty_insider = analyze_insider_activity("NO_SUCH_TICKER", PROJECT_ROOT)

assert empty_news["article_count"] == 0
assert empty_market["row_count"] == 0
assert empty_insider["transaction_count"] == 0

print("Specialist tests passed.")

## 6. Synthesis Agent

The Synthesis Agent combines the three specialist results into **risks** and **catalysts**, using simple rules over news categories, average daily return, and insider acquired/disposed counts.

In [ ]:
synthesis_result = synthesize_research(
    TICKER,
    news_result,
    market_result,
    insider_result,
)

print("Risks:")
for risk in synthesis_result["risks"]:
    print(f"- {risk}")

print("\nCatalysts:")
for catalyst in synthesis_result["catalysts"]:
    print(f"- {catalyst}")

print(f"\nEvidence used: {synthesis_result['summary']}")

In [ ]:
# Tests: synthesis rules fire correctly on controlled inputs.
bearish = synthesize_research(
    "TEST",
    {"categories": {"macro_market": 2}, "article_count": 2},
    {"average_daily_return": -0.01, "row_count": 10},
    {"acquired_transactions": 1, "disposed_transactions": 4, "transaction_count": 5},
)
assert len(bearish["risks"]) == 3
assert bearish["summary"] == {
    "news_articles": 2,
    "market_rows": 10,
    "insider_transactions": 5,
}

bullish = synthesize_research(
    "TEST",
    {"categories": {"analyst_investor": 1}},
    {"average_daily_return": 0.02},
    {"acquired_transactions": 3, "disposed_transactions": 0},
)
assert bullish["risks"] == []
assert len(bullish["catalysts"]) == 3

# Missing evidence produces empty lists, not an error.
empty = synthesize_research("TEST", {}, {}, {})
assert empty["risks"] == [] and empty["catalysts"] == []

print("Synthesis tests passed.")

## 7. Full Orchestration

`orchestrate_research()` runs the whole workflow in one call: the Planner checks the evidence and builds the plan, the Router assigns each step, and **only the specialist agents that were assigned a step are called**. Skipped agents return a short message instead. The Synthesis Agent then combines the results. Each executed step records which agent handled it and why.

In [ ]:
result = orchestrate_research(TICKER, PROJECT_ROOT)

display_orchestration(result)

In [ ]:
pd.DataFrame(
    [
        {
            "step_id": step["step_id"],
            "task": step["task"],
            "agent": step["agent"],
            "reason": step["reason"],
        }
        for step in result["executed_steps"]
    ]
)

In [ ]:
# Tests: the end-to-end run matches the individual steps above.
assert result["ticker"] == TICKER
assert result["plan"] == plan
assert result["routing_decisions"] == routing_decisions
assert len(result["executed_steps"]) == len(plan["research_steps"])

for step in result["executed_steps"]:
    assert step["result"] is result["specialist_results"][step["agent"]]

assert result["final_synthesis"]["risks"] == synthesis_result["risks"]
assert result["final_synthesis"]["catalysts"] == synthesis_result["catalysts"]

# Only the agents the router selected are called.
routed_agents = {d["agent"] for d in routing_decisions} - {"synthesis_agent"}
assert set(result["agents_called"]) == routed_agents

# For a ticker with no data, no specialist agent is called.
empty_run = orchestrate_research("NO_SUCH_TICKER", PROJECT_ROOT)
assert empty_run["agents_called"] == []
assert all(r.get("skipped") for name, r in empty_run["specialist_results"].items()
           if name != "synthesis_agent")

print("Orchestration tests passed.")

## 8. Run for Every Available Ticker

Confirms the workflow runs end to end for each ticker in the processed data.

In [ ]:
summary_rows = []

for ticker in available_tickers:
    run = orchestrate_research(ticker, PROJECT_ROOT)
    evidence = run["final_synthesis"]["summary"]
    summary_rows.append(
        {
            "ticker": ticker,
            "news_articles": evidence["news_articles"],
            "market_rows": evidence["market_rows"],
            "insider_transactions": evidence["insider_transactions"],
            "risks": len(run["final_synthesis"]["risks"]),
            "catalysts": len(run["final_synthesis"]["catalysts"]),
        }
    )

pd.DataFrame(summary_rows)

## 9. Handoff to Evaluator–Optimizer and Memory

The Evaluator–Optimizer workflow should call the orchestrator rather than re-run the agents:

```python
from src.orchestrator import orchestrate_research

result = orchestrate_research(ticker, PROJECT_ROOT)
```

Useful fields in `result`:

| Key | Contents |
|---|---|
| `final_synthesis["risks"]` | List of risk statements |
| `final_synthesis["catalysts"]` | List of catalyst statements |
| `final_synthesis["summary"]` | Evidence counts (news, market, insider) |
| `specialist_results` | Full output of each specialist, for evaluating evidence coverage |
| `executed_steps` | Step-by-step plan, agent, and reason — useful to store as run memory |

This notebook does not implement self-reflection, refinement, or cross-run memory.